[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tippe_top/tippe_top.ipynb)

# Chiral Tippe Tops

This example simulates a chiral hollow tippe top released on a flat table with opposite spins and tracks which spin direction inverts onto its stem. The body is modelled as an outer triaxial ellipsoid carved by an offset, tilted ellipsoidal cavity and joined to a rounded stem and spherical tip of uniform density. Because the cavity turns the principal axes of the mass distribution away from the axes of the outer shell, sliding friction at the shell and tip couples spinning to rocking differently for the two spin directions, sending one top onto its stem while leaving the other upright.

Each constituent ellipsoid is represented as a quadric map from points to polar planes, so `(parts[:, None](samples) & samples) <= 0` carves the hollow solid from uniform sample boxes while the same quadrics render directly on the GPU. Summing `(masses * samples).regressive(samples.commutator(Rate))` over the enclosed samples assembles the full rigid-body inertia map ([Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)) without splitting linear and angular equations or shifting to the centre of mass. Stepping a motor `placement` through free rigid motion and frictional contact updates both contact surfaces together, and `z & (placement >> up)` reads the stem's vertical inclination.

In matrix Newton–Euler notation, each displaced and rotated ellipsoid is written in homogeneous coordinates as $\tilde{\mathbf x}^{\mathsf T} M^{-\mathsf T} \operatorname{diag}(a^{-2}, b^{-2}, c^{-2}, -1) M^{-1} \tilde{\mathbf x} \leq 0$, and the mass samples first determine $\mathbf r_{\mathrm{cm}} = M_{\mathrm{tot}}^{-1}\sum_k m_k \mathbf r_k$ before shifting to the $3 \times 3$ rotational inertia tensor $I_{\mathrm{cm}} = \sum_k m_k(\|\Delta\mathbf r_k\|^2 I_3 - \Delta\mathbf r_k \Delta\mathbf r_k^{\mathsf T})$ with $\Delta\mathbf r_k = \mathbf r_k - \mathbf r_{\mathrm{cm}}$. Motion then integrates separate translational and rotational states $(\mathbf r_{\mathrm{cm}}, \mathbf v_{\mathrm{cm}}, R, \boldsymbol\omega)$ under $\dot R = R[\boldsymbol\omega]_\times$, $M_{\mathrm{tot}}\dot{\mathbf v}_{\mathrm{cm}} = \mathbf f_g + \mathbf f_c$, and $I_{\mathrm{cm}}\dot{\boldsymbol\omega} + \boldsymbol\omega \times (I_{\mathrm{cm}}\boldsymbol\omega) = R^{\mathsf T}(\mathbf r_c - \mathbf r_{\mathrm{cm}}) \times R^{\mathsf T}\mathbf f_c$.

In [ ]:
# Install numga and its examples when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import jax
import numpy as np

from numga import stack
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from examples.mechanics import lie_integrators as lie
from examples.mechanics.spinning_top import core as contact
from examples.mechanics.tippe_top import render
from examples.util.animation import save_figure, show_states

jax.config.update("jax_enable_x64", True)
mv = JaxContext(ga, dtype=np.float64).multivector
Point = ga.gatype.antivector()                                               # Antivector-
Plane = ga.gatype.vector()                                                   # Vector+
Rate = ga.gatype.bivector()                                                  # Bivector+
Forque = ga.gatype.antibivector()                                            # Antibivector-
Motor = ga.gatype.rotor()                                                    # Rotor+
Quadric = ga.gatype((Plane, Point))                                          # Plane+ <- Point-
Inertia = ga.gatype((Forque, Rate))                                          # Forque- <- Rate+
origin = mv.w.dual()                                                         # [] Point-
up = mv.z.dual()                                                             # [] Point-
axes = stack((mv.x, mv.y, mv.z))                                             # [3] Plane+
ground = (mv.z * (mv.w & Point) + mv.w * (mv.z & Point)) / 2                 # [] Plane+ <- Point-

## Hollow body and mass distribution

An ellipsoidal cavity is offset and turned inside an outer triaxial shell, breaking through the top opening while leaving the lower contact surface intact. Because the cavity and shell have unequal horizontal semiaxes oriented at an angle to each other, the enclosed mass distribution favours a rocking plane turned away from the shell's geometric axes. An ellipsoidal stem and spherical tip complete one rigid solid of uniform density whose overlapping regions count once.

In [ ]:
# Outer shell at the origin, offset tilted cavity breaking through its top, stem, and spherical tip.
shell_semiaxes = np.array([1.05, 0.95, 1.01])
cavity_centre = mv.point([-0.05, -0.04, 0.47])                               # [] Point-
cavity_semiaxes = np.array([0.72, 0.65, 0.59])
cavity_angle = 0.16
cavity_tilt = 0.07
stem_centre = mv.point([0, 0, 0.21])                                         # [] Point-
stem_semiaxes = np.array([0.18, 0.18, 1.02])
tip_centre = mv.point([0, 0, 1.23])                                          # [] Point-
tip_radius = 0.20
sample_spacing = 0.025
sample_lower = np.array([-1.125, -1.125, -1.125])
sample_upper = np.array([1.125, 1.125, 1.525])

# Equal-volume boxes cover the solid so each midpoint carries the material in its box.
sample_axes = [np.arange(low, high, sample_spacing) + sample_spacing / 2
               for low, high in zip(sample_lower, sample_upper)]
grid_points = np.stack(np.meshgrid(*sample_axes, indexing="ij"), axis=-1).reshape(-1, 3)
n_samples = len(grid_points)
samples = mv.point(grid_points)                                              # [n_samples] Point-

In [ ]:
def ellipsoid(centre: Point, frame: Plane, semiaxes: np.ndarray) -> Quadric:
    """Planes through `centre` along `frame` (`[3] Plane+`) scaled by `semiaxes` form the ellipsoid's surface quadric (`[] Plane+ <- Point-`)."""
    axial_planes = frame - mv.w * (frame & centre)                           # [3] Plane+
    return (axial_planes * (axial_planes & Point) / semiaxes**2).sum(axis=-1) - mv.w * (mv.w & Point)


@jax.jit
def mass_distribution(parts: Quadric, samples: Point, sample_volume: float) -> tuple[Point, Inertia]:
    """Uniform material inside the carved shell, stem, and tip (`[4] Plane+ <- Point-`) determines the mass point (`[] Point-`) and inertia map (`[] Forque- <- Rate+`)."""
    shell_mask, cavity_mask, stem_mask, tip_mask = (parts[:, None](samples) & samples) <= 0
    material_mask = (shell_mask & ~cavity_mask) | stem_mask | tip_mask       # [...]
    masses = material_mask * sample_volume                                   # [...]
    mass = (samples * masses).sum(axis=-1)                                   # [] Point-
    # Each mass sample joins its velocity under a rate to its position, contributing to the body's momentum.
    inertia = (masses * samples).regressive(samples.commutator(Rate)).sum(axis=-1)  # [] Forque- <- Rate+
    return mass, inertia


cavity_turn = (mv.yz * (-cavity_tilt / 2)).exp() * (mv.xy * (-cavity_angle / 2)).exp()  # [] Motor+
cavity_axes = cavity_turn >> axes                                            # [3] Plane+
shell = ellipsoid(origin, axes, shell_semiaxes)                              # [] Plane+ <- Point-
cavity = ellipsoid(cavity_centre, cavity_axes, cavity_semiaxes)              # [] Plane+ <- Point-
stem = ellipsoid(stem_centre, axes, stem_semiaxes)                           # [] Plane+ <- Point-
tip = ellipsoid(tip_centre, axes, tip_radius * np.ones(3))                   # [] Plane+ <- Point-
parts = stack((shell, cavity, stem, tip))                                    # [4] Plane+ <- Point-
contacts = stack((shell, tip))                                               # [2] Plane+ <- Point-
mass, inertia = mass_distribution(parts, samples, sample_spacing**3)         # [] Point-, Forque- <- Rate+
save_figure(render.setup_figure(parts, mass), "tippe_top_construction");

Alternating terracotta and peach octants mark body-fixed planes on the carved shell so its rotation remains visible during spinning, while the gold stem and spherical tip rise out of the tilted interior cavity.

## Opposite-spin launches and contact stepping

The outer shell and the spherical tip act as the two candidate contact surfaces of the rigid body, while the recessed stem stays clear of their support envelope. Releasing the body from the same slight lean with equal and opposite spins about the vertical line through the mass centre isolates the chiral coupling between sliding friction and rocking.

In [ ]:
spin_speed = 13.0
spins = spin_speed * np.array([-1, 1])
lean = 0.1
gravity = 1.0
dynamic_friction = 0.35
static_friction = 1.2 * dynamic_friction
step = 0.004
frame_interval = 0.16
n_steps_per_frame = round(frame_interval / step)
run_time = 80.0
n_states = round(run_time / frame_interval) + 1
flip_start, flip_end = 53.0, 65.0
first_frame = round(flip_start / frame_interval)
n_frames = round((flip_end - flip_start) / frame_interval)
frame_duration_ms = 40

In [ ]:
def release_states(contacts: Quadric, mass: Point, spins: np.ndarray, lean: float) -> tuple[Motor, Rate]:
    """Tilts the body by `lean` without table penetration and assigns `spins` (`[...]`) about the vertical line through `mass`."""
    tilt = (mv.xz * (lean / 2)).exp()                                        # [] Motor+
    body_up = tilt << up                                                     # [] Point-
    supports = contact.lowest_point(contacts, body_up)                       # [2] Point-
    placed_supports = tilt >> supports                                       # [2] Point-
    height = (mv.z & placed_supports).min(axis=0)                            # [] Scalar+
    lift = (-up * origin.inverse() * height / 2).exp()                       # [] Motor+
    motor = (lift * tilt).broadcast_to(spins.shape)                          # [...] Motor+
    centre = mass.normalized()                                               # [] Point-
    spin_axis = (mv.x - mv.w * (mv.x & centre)) ^ (mv.y - mv.w * (mv.y & centre))  # [] Rate+
    return motor.retyped(Motor), (spin_axis * spins).retyped(Rate)


motor, rate = release_states(contacts, mass, spins, lean)                    # [2] Motor+, Rate+

Each substep integrates free rigid motion under gravity on the motor group and projects active shell and tip contacts against the table with Coulomb friction, deriving the updated rate from the corrected placement. Stepping the full run and rendering the inversion window takes about ten seconds.

In [ ]:
@partial(jax.jit, static_argnames=("n_steps",))
def frame_step(contacts: Quadric, mass: Point, inertia: Inertia, motor: Motor, rate: Rate,
               gravity: float, static_friction: float, dynamic_friction: float,
               dt: float, n_steps: int) -> tuple[Motor, Rate]:
    """Advances `motor` (`[...] Motor+`) and `rate` (`[...] Rate+`) across `n_steps` free-motion and contact substeps."""
    inverse_inertia = inertia.inverse()                                      # [] Rate+ <- Forque-

    def gravity_forque(placement: Motor, velocity: Rate) -> Forque:
        gravity_force = placement << (-up * gravity)                         # [...] Point-
        return mass & gravity_force

    def substep(step_index: int, state: tuple[Motor, Rate]) -> tuple[Motor, Rate]:
        prior_motor, velocity = state
        predicted_motor, _ = lie.explicit_rk4(prior_motor, velocity, inertia, inverse_inertia, dt, gravity_forque)
        next_motor, next_rate = contact.project_contacts(
            prior_motor, predicted_motor.normalized(), inverse_inertia,
            contacts, ground, static_friction, dynamic_friction, dt,
        )
        return next_motor.retyped(Motor), next_rate.retyped(Rate)

    return jax.lax.fori_loop(0, n_steps, substep, (motor, rate))


def motion(contacts: Quadric, mass: Point, inertia: Inertia, motor: Motor, rate: Rate,
           gravity: float, static_friction: float, dynamic_friction: float,
           dt: float, n_states: int, n_steps_per_frame: int) -> Iterator[tuple[Motor, Rate]]:
    """Yields `n_states` pairs of placement (`[...] Motor+`) and rate (`[...] Rate+`) separated by `n_steps_per_frame` substeps."""
    yield motor, rate
    for _ in range(n_states - 1):
        motor, rate = frame_step(contacts, mass, inertia, motor, rate, gravity,
                                 static_friction, dynamic_friction, dt, n_steps_per_frame)
        yield motor, rate


recorded_states = show_states(
    motion(contacts, mass, inertia, motor, rate, gravity, static_friction, dynamic_friction,
           step, n_states, n_steps_per_frame),
    render.frames, parts, mass, first_frame, n_frames,
    name="tippe_top_opposite_spins", duration_ms=frame_duration_ms, n_frames=n_frames,
);

The left panel follows the negative-spin launch and the right panel follows the positive-spin launch across the flip window, with each camera tracking horizontal drift while averaging out rapid rocking. On the left, sliding friction tips the hollow shell past horizontal until the spherical stem tip takes over contact and holds the inverted top upright, while the right launch continues spinning upright on its shell.

## Stem inclination history

Pairing the world vertical plane with the placed stem direction `z & (placement >> up)` tracks the cosine of the stem's inclination across the entire run from the same recorded trajectory.

In [ ]:
placements = stack([placement for placement, _ in recorded_states])          # [n_states, 2] Motor+
stem_directions = placements >> up                                           # [n_states, 2] Point-
vertical = mv.z & stem_directions                                            # [n_states, 2] Scalar+
times = np.arange(n_states) * frame_interval
save_figure(render.inclination_figure(times, vertical), "tippe_top_inclination");

The orange curve tracks the negative-spin launch and the blue curve tracks the positive-spin launch from upright at the top of the axis through horizontal at zero to inverted at the bottom. Growing nutation swings carry the orange trajectory past the horizontal plane into steady inverted spin on the spherical tip, whereas the blue trajectory damps back toward its upright release angle.

In [ ]:
# checks
assert (mv.w & mass) > 0
assert ((stack((mv.x, mv.y)) & mass.normalized()).abs() > 0).all()
assert ((mv.xy & inertia(mv.yz)) - (mv.yz & inertia(mv.xy))).abs() < 1e-12
assert ((motor.norm() - 1).abs() < 1e-12).all()
assert (rate[0] + rate[1]).scalar_norm_squared() < 1e-12
assert ((placements[-1].norm() - 1).abs() < 1e-12).all()
assert (vertical[0, 0] - vertical[0, 1]).abs() < 1e-12
assert vertical[-1, 0] < -0.8
assert vertical[-1, 1] > 0.8